# 🛡️ NSFW Image Classifier — ResNet50 + Knowledge Distillation

Model klasifikasi biner (safe/nsfw) menggunakan **ResNet50 Transfer Learning** + **nsfw_detector** sebagai Teacher Model untuk **Auto-Labeling** dan **Knowledge Distillation**.

---

### Pipeline:
- **Phase A:** Auto-label gambar mentah menggunakan nsfw_detector (Teacher)
- **Fase 2:** Validasi & pembersihan data
- **Fase 3:** Preprocessing & augmentasi
- **Fase 4:** Arsitektur ResNet50 (Student)
- **Phase B:** Training dengan Knowledge Distillation
- **Fase 6-7:** Evaluasi & inferensi (ditunda)

---
## 📦 Cell 1 — Setup & Imports

In [ ]:
import os
import sys
import shutil
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

# ==========================================
# KONFIGURASI
# ==========================================
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
ALPHA = 0.7  # Bobot Knowledge Distillation: 0.7 hard label + 0.3 teacher

# Path
DATA_DIR = 'data'              # Folder data utama
RAW_DIR = 'data/raw'           # Gambar mentah (belum dilabeli)
SAFE_DIR = 'data/safe'         # Output: gambar safe
NSFW_DIR = 'data/nsfw'         # Output: gambar nsfw
TEACHER_CACHE = 'teacher_predictions.npy'  # Cache soft labels

# Path ke model nsfw_detector (ubah sesuai lokasi)
NSFW_DETECTOR_PATH = '../nsfw_detector-1.1.1'
TEACHER_MODEL_PATH = 'mobilenet_v2_140_224'  # Folder SavedModel

# Deteksi GPU
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU terdeteksi: {[g.name for g in gpus]}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("⚠️  Tidak ada GPU, menggunakan CPU")

print(f"\n📐 Konfigurasi:")
print(f"   Image Size       : {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch Size       : {BATCH_SIZE}")
print(f"   Epochs           : {EPOCHS}")
print(f"   Learning Rate    : {LEARNING_RATE}")
print(f"   Distillation α   : {ALPHA} (hard) / {1-ALPHA:.1f} (soft)")

TensorFlow version: 2.19.0
⚠️  Tidak ada GPU, menggunakan CPU

📐 Konfigurasi:
   Image Size       : 224x224
   Batch Size       : 32
   Epochs           : 20
   Learning Rate    : 0.0001
   Distillation α   : 0.7 (hard) / 0.3 (soft)


---
## 🏷️ Phase A — Auto-Labeling dengan nsfw_detector

Menggunakan model **pre-trained nsfw_detector** (dilatih pada 60GB+ data, akurasi 93%) untuk:
1. Mengklasifikasi gambar mentah dari `data/raw/`
2. Mapping 5 kelas → 2 kelas: `porn + hentai + sexy` → **nsfw**, `neutral + drawings` → **safe**
3. Memindahkan gambar ke folder `data/safe/` atau `data/nsfw/`
4. Menyimpan probabilitas teacher untuk Knowledge Distillation

In [ ]:
# ==========================================
# A1: LOAD TEACHER MODEL (nsfw_detector)
# ==========================================

# Tambahkan path nsfw_detector ke sys.path
sys.path.insert(0, os.path.abspath(NSFW_DETECTOR_PATH))
from nsfw_detector import predict as nsfw_predict

# Load model teacher
print("🔄 Loading nsfw_detector teacher model...")
teacher_model = nsfw_predict.load_model(TEACHER_MODEL_PATH)
print("✅ Teacher model loaded!")

# Quick sanity check — daftar kelas teacher
print("\n📋 Teacher model output:")
print("   5 kelas: drawings, hentai, neutral, porn, sexy")
print("   Mapping → nsfw = (porn + hentai + sexy)")
print("           → safe = (neutral + drawings)")

In [ ]:
# ==========================================
# A2: AUTO-LABEL PIPELINE
# ==========================================

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

def auto_label_images(raw_dir, safe_dir, nsfw_dir, teacher_model, threshold=0.5):
    """
    Klasifikasi gambar dari raw_dir menggunakan teacher model,
    lalu pindahkan ke safe_dir atau nsfw_dir.

    Returns:
        teacher_probs: dict {filename: nsfw_probability}
        stats: dict dengan statistik labeling
    """
    # Pastikan folder output ada
    os.makedirs(safe_dir, exist_ok=True)
    os.makedirs(nsfw_dir, exist_ok=True)

    # Scan gambar valid
    image_files = []
    for fname in os.listdir(raw_dir):
        ext = os.path.splitext(fname)[1].lower()
        if ext in VALID_EXTENSIONS:
            image_files.append(fname)

    if not image_files:
        print(f"❌ Tidak ada gambar ditemukan di '{raw_dir}'")
        print(f"   Pastikan gambar (.jpg/.jpeg/.png) ada di folder tersebut.")
        return None, None

    print(f"📁 Ditemukan {len(image_files)} gambar di '{raw_dir}'")
    print(f"🔍 Memulai klasifikasi dengan teacher model...\n")

    teacher_probs = {}  # {filename: nsfw_probability}
    safe_count = 0
    nsfw_count = 0
    errors = []

    # Proses per batch untuk efisiensi
    batch_size = 32
    for i in range(0, len(image_files), batch_size):
        batch_files = image_files[i:i+batch_size]
        batch_paths = [os.path.join(raw_dir, f) for f in batch_files]

        try:
            # Klasifikasi batch
            results = nsfw_predict.classify(teacher_model, batch_paths, image_dim=IMG_SIZE)

            for fpath, probs in results.items():
                fname = os.path.basename(fpath)

                # Hitung skor nsfw (porn + hentai + sexy)
                nsfw_score = probs.get('porn', 0) + probs.get('hentai', 0) + probs.get('sexy', 0)
                safe_score = probs.get('neutral', 0) + probs.get('drawings', 0)

                # Simpan probabilitas teacher (untuk distillation nanti)
                teacher_probs[fname] = nsfw_score

                # Tentukan label dan pindahkan file
                src = os.path.join(raw_dir, fname)
                if nsfw_score > threshold:
                    dst = os.path.join(nsfw_dir, fname)
                    nsfw_count += 1
                else:
                    dst = os.path.join(safe_dir, fname)
                    safe_count += 1

                shutil.copy2(src, dst)  # copy (tidak hapus dari raw)

        except Exception as e:
            errors.append((batch_files, str(e)))

        # Progress
        done = min(i + batch_size, len(image_files))
        print(f"   Progres: {done}/{len(image_files)} gambar", end='\r')

    print(f"\n\n✅ AUTO-LABELING SELESAI!")
    print(f"{'=' * 45}")
    print(f"   Safe  : {safe_count} gambar → data/safe/")
    print(f"   NSFW  : {nsfw_count} gambar → data/nsfw/")
    print(f"   Error : {len(errors)} batch")
    print(f"{'=' * 45}")

    if errors:
        print(f"\n⚠️  Error pada:")
        for files, err in errors[:3]:
            print(f"   {files[:2]}... : {err}")

    stats = {'safe': safe_count, 'nsfw': nsfw_count, 'errors': len(errors)}
    return teacher_probs, stats


# ==========================================
# JALANKAN AUTO-LABELING
# ==========================================
teacher_probs, label_stats = auto_label_images(
    RAW_DIR, SAFE_DIR, NSFW_DIR, teacher_model, threshold=0.5
)

# Simpan teacher probabilities untuk Knowledge Distillation
if teacher_probs is not None:
    np.save(TEACHER_CACHE, teacher_probs)
    print(f"\n💾 Teacher predictions disimpan ke '{TEACHER_CACHE}'")
    print(f"   Total: {len(teacher_probs)} entries")

In [ ]:
# ==========================================
# A3: REVIEW GAMBAR PALING TIDAK PASTI
# ==========================================
# Tampilkan gambar yang paling mendekati threshold 0.5
# (paling sulit diputuskan oleh teacher) untuk review manual

if teacher_probs is not None:
    # Urutkan berdasarkan jarak ke threshold 0.5
    sorted_by_uncertainty = sorted(
        teacher_probs.items(),
        key=lambda x: abs(x[1] - 0.5)
    )

    # Ambil 20 gambar paling tidak pasti
    uncertain = sorted_by_uncertainty[:20]

    print(f"🔍 TOP 20 GAMBAR PALING TIDAK PASTI (mendekati threshold 0.5):")
    print(f"{'=' * 60}")
    print(f"{'No':>3} | {'File':<30} | {'NSFW Score':>10} | {'Label':>6}")
    print(f"{'-' * 60}")
    for idx, (fname, score) in enumerate(uncertain, 1):
        label = 'NSFW' if score > 0.5 else 'SAFE'
        marker = '⚠️' if abs(score - 0.5) < 0.1 else '  '
        print(f"{idx:>3} | {fname:<30} | {score:>9.4f} | {label:>6} {marker}")
    print(f"{'=' * 60}")
    print(f"\n⚠️  Gambar dengan skor mendekati 0.5 mungkin perlu review manual.")
    print(f"   Pindahkan secara manual jika label yang diberikan salah.")

    # Visualisasi distribusi skor teacher
    scores = list(teacher_probs.values())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    axes[0].hist(scores, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold')
    axes[0].set_title('Distribusi Skor NSFW (Teacher)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('NSFW Probability')
    axes[0].set_ylabel('Jumlah Gambar')
    axes[0].legend()

    # Confidence zones
    high_conf_safe = sum(1 for s in scores if s < 0.2)
    low_conf = sum(1 for s in scores if 0.2 <= s <= 0.8)
    high_conf_nsfw = sum(1 for s in scores if s > 0.8)

    zones = ['Safe\n(< 0.2)', 'Uncertain\n(0.2 - 0.8)', 'NSFW\n(> 0.8)']
    counts = [high_conf_safe, low_conf, high_conf_nsfw]
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    axes[1].bar(zones, counts, color=colors, edgecolor='white', linewidth=2)
    axes[1].set_title('Zona Kepercayaan Teacher', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Jumlah Gambar')
    for i, c in enumerate(counts):
        axes[1].text(i, c + max(counts)*0.02, str(c), ha='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

---
## 🗂️ Cell 2 — Fase 2: Validasi Data

Setelah auto-labeling, validasi integritas file di `data/safe/` dan `data/nsfw/`.

In [ ]:
# ==========================================
# VALIDASI DATA SETELAH AUTO-LABELING
# ==========================================

def validate_labeled_data(data_dir):
    """Validasi file gambar yang sudah dilabeli."""
    stats = {}
    corrupted = []

    for class_name in sorted(os.listdir(data_dir)):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir) or class_name == 'raw':
            continue

        valid = 0
        for fname in os.listdir(class_dir):
            fpath = os.path.join(class_dir, fname)
            ext = os.path.splitext(fname)[1].lower()
            if ext not in VALID_EXTENSIONS:
                continue
            try:
                img = Image.open(fpath)
                img.verify()
                valid += 1
            except Exception as e:
                corrupted.append(fpath)

        stats[class_name] = valid

    return stats, corrupted

print("🔍 Memvalidasi data yang sudah dilabeli...\n")
stats, corrupted = validate_labeled_data(DATA_DIR)

print("📊 STATISTIK DATASET (Post Auto-Labeling):")
print("=" * 40)
total = 0
for cls, count in stats.items():
    print(f"   Kelas '{cls}': {count} gambar")
    total += count
print(f"   {'─' * 30}")
print(f"   TOTAL: {total} gambar valid")

if corrupted:
    print(f"\n❌ {len(corrupted)} file corrupt — menghapus...")
    for f in corrupted:
        os.remove(f)
        print(f"   Dihapus: {os.path.basename(f)}")
else:
    print("\n✅ Semua file valid!")

---
## 🔄 Cell 3 — Fase 3: Preprocessing & Augmentation

- Resize ke 224×224
- Normalisasi ImageNet (`preprocess_input`)
- Augmentasi: Horizontal Flip, Rotation, Brightness, Zoom

In [ ]:
# ==========================================
# IMAGE DATA GENERATORS
# ==========================================

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    fill_mode='constant',
    cval=0
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=42,
    classes=['nsfw', 'safe']  # Eksplisit: nsfw=0, safe=1
)

print(f"\n📋 Class Mapping: {train_generator.class_indices}")
print(f"   Total sampel: {train_generator.samples}")
print(f"   Steps per epoch: {len(train_generator)}")

In [ ]:
# ==========================================
# VISUALISASI SAMPEL AUGMENTASI
# ==========================================

batch_images, batch_labels = next(train_generator)
class_names = {v: k for k, v in train_generator.class_indices.items()}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sampel Gambar (Setelah Augmentasi)', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(batch_images):
        img = batch_images[i].copy()
        # Denormalisasi dari preprocess_input (caffe mode)
        img[..., 0] += 103.939
        img[..., 1] += 116.779
        img[..., 2] += 123.68
        img = img[..., ::-1]  # BGR -> RGB
        img = np.clip(img, 0, 255).astype('uint8')

        label = class_names[int(batch_labels[i])]
        color = 'green' if label == 'safe' else 'red'
        ax.imshow(img)
        ax.set_title(f'{label}', color=color, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 🧠 Cell 4 — Fase 4: Arsitektur ResNet50 (Student Model)

1. ResNet50 pre-trained (ImageNet) — **frozen**
2. Custom head: `GAP → Dense(256, ReLU) → Dropout(0.5) → Dense(1, Sigmoid)`

In [ ]:
# ==========================================
# LOAD BASE MODEL: ResNet50
# ==========================================
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze seluruh layer bawaan
for layer in base_model.layers:
    layer.trainable = False

print(f"✅ ResNet50 base model loaded (frozen)")
print(f"   Total layers: {len(base_model.layers)}")
print(f"   Output shape: {base_model.output_shape}")

# ==========================================
# CUSTOM CLASSIFICATION HEAD
# ==========================================
x = base_model.output
x = GlobalAveragePooling2D(name='gap')(x)
x = Dense(256, activation='relu', name='dense_256')(x)
x = Dropout(0.5, name='dropout')(x)
output = Dense(1, activation='sigmoid', name='output')(x)

student_model = Model(
    inputs=base_model.input,
    outputs=output,
    name='NSFW_ResNet50_Student'
)

# Ringkasan
total_params = student_model.count_params()
trainable = sum(tf.keras.backend.count_params(w) for w in student_model.trainable_weights)
frozen = total_params - trainable

print(f"\n📊 ARSITEKTUR STUDENT MODEL:")
print(f"{'=' * 45}")
print(f"   Total Parameters     : {total_params:>12,}")
print(f"   Trainable (head)     : {trainable:>12,}")
print(f"   Frozen (ResNet50)    : {frozen:>12,}")
print(f"   Rasio Trainable      : {trainable/total_params*100:.2f}%")
print(f"{'=' * 45}")

---
## 🎓 Phase B — Training dengan Knowledge Distillation

**Konsep:** Model student (ResNet50) belajar dari **dua sumber**:
1. **Hard Labels** (0/1) — dari folder safe/nsfw
2. **Soft Labels** (probabilitas) — dari teacher model (nsfw_detector)

**Loss Function:**
```
total_loss = α × BCE(hard_label, prediction) + (1-α) × BCE(teacher_prob, prediction)
```

Keuntungan: Student mendapat informasi **nuansa** dari teacher — misalnya gambar bikini yang "80% safe" vs pakaian biasa yang "99% safe". Ini membuat student lebih pintar daripada hanya belajar dari label biner.

In [ ]:
# ==========================================
# B1: PERSIAPAN TEACHER SOFT LABELS
# ==========================================

# Load cached teacher predictions
if os.path.exists(TEACHER_CACHE):
    teacher_probs_loaded = np.load(TEACHER_CACHE, allow_pickle=True).item()
    print(f"✅ Teacher predictions loaded: {len(teacher_probs_loaded)} entries")
else:
    print("⚠️  Teacher cache tidak ditemukan. Jalankan Phase A terlebih dahulu.")
    teacher_probs_loaded = {}

# Buat mapping filename -> teacher soft label
# Untuk generator, kita perlu mapping berdasarkan filepath
def get_teacher_label(filepath, teacher_dict):
    """
    Ambil soft label dari teacher berdasarkan filename.
    Teacher score = probabilitas NSFW.
    Generator kita: nsfw=0, safe=1
    Jadi soft label = 1 - nsfw_score (karena safe=1)
    """
    fname = os.path.basename(filepath)
    if fname in teacher_dict:
        nsfw_score = teacher_dict[fname]
        return 1.0 - nsfw_score  # Convert ke skala safe (match generator)
    return None  # Fallback: gunakan hard label saja

# Pre-compute soft labels untuk semua file dalam generator
soft_labels = {}
for fpath in train_generator.filepaths:
    fname = os.path.basename(fpath)
    if fname in teacher_probs_loaded:
        soft_labels[fpath] = 1.0 - teacher_probs_loaded[fname]

coverage = len(soft_labels) / max(len(train_generator.filepaths), 1) * 100
print(f"\n📊 Cakupan soft labels: {len(soft_labels)}/{len(train_generator.filepaths)} ({coverage:.1f}%)")
print(f"   Gambar tanpa soft label akan menggunakan hard label saja.")

In [ ]:
# ==========================================
# B2: KNOWLEDGE DISTILLATION TRAINING LOOP
# ==========================================

# Compile model
optimizer = Adam(learning_rate=LEARNING_RATE)
bce_loss_fn = tf.keras.losses.BinaryCrossentropy()

# Tracking metrik
history = {
    'loss': [], 'hard_loss': [], 'soft_loss': [], 'accuracy': []
}

best_loss = float('inf')
best_epoch = 0

print(f"🏋️ KNOWLEDGE DISTILLATION TRAINING")
print(f"{'=' * 55}")
print(f"   Alpha (hard weight) : {ALPHA}")
print(f"   1-Alpha (soft weight): {1-ALPHA:.1f}")
print(f"   Epochs              : {EPOCHS}")
print(f"   Steps/epoch         : {len(train_generator)}")
print(f"{'=' * 55}\n")

for epoch in range(EPOCHS):
    epoch_loss = []
    epoch_hard_loss = []
    epoch_soft_loss = []
    epoch_correct = 0
    epoch_total = 0

    # Reset generator di awal setiap epoch
    train_generator.reset()

    for step in range(len(train_generator)):
        # Ambil batch
        x_batch, y_hard = train_generator[step]

        # Bangun soft labels untuk batch ini
        batch_start = step * BATCH_SIZE
        batch_filepaths = train_generator.filepaths[batch_start:batch_start + len(x_batch)]

        y_soft = np.array([
            soft_labels.get(fp, y_hard[i])  # fallback ke hard label
            for i, fp in enumerate(batch_filepaths)
        ], dtype=np.float32)

        with tf.GradientTape() as tape:
            # Forward pass
            y_pred = student_model(x_batch, training=True)
            y_pred = tf.squeeze(y_pred)

            # Hard loss (dari label folder)
            hard_loss = bce_loss_fn(
                tf.cast(y_hard, tf.float32), y_pred
            )

            # Soft loss (dari teacher probabilities)
            soft_loss = bce_loss_fn(
                tf.cast(y_soft, tf.float32), y_pred
            )

            # Combined loss
            total_loss = ALPHA * hard_loss + (1 - ALPHA) * soft_loss

        # Backward pass
        grads = tape.gradient(total_loss, student_model.trainable_weights)
        optimizer.apply_gradients(zip(grads, student_model.trainable_weights))

        # Tracking
        epoch_loss.append(float(total_loss))
        epoch_hard_loss.append(float(hard_loss))
        epoch_soft_loss.append(float(soft_loss))

        preds_binary = (y_pred.numpy() > 0.5).astype(int)
        epoch_correct += np.sum(preds_binary == y_hard.astype(int))
        epoch_total += len(y_hard)

    # Epoch metrics
    avg_loss = np.mean(epoch_loss)
    avg_hard = np.mean(epoch_hard_loss)
    avg_soft = np.mean(epoch_soft_loss)
    acc = epoch_correct / max(epoch_total, 1)

    history['loss'].append(avg_loss)
    history['hard_loss'].append(avg_hard)
    history['soft_loss'].append(avg_soft)
    history['accuracy'].append(acc)

    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_epoch = epoch + 1
        student_model.save('best_model.keras')
        save_marker = ' ✅ SAVED'
    else:
        save_marker = ''

    print(
        f"Epoch {epoch+1:>2}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f} (H:{avg_hard:.4f} S:{avg_soft:.4f}) | "
        f"Acc: {acc*100:.1f}%{save_marker}"
    )

print(f"\n{'=' * 55}")
print(f"✅ Training selesai!")
print(f"   Best model di epoch {best_epoch} (loss: {best_loss:.4f})")
print(f"   Tersimpan di: best_model.keras")

In [ ]:
# ==========================================
# VISUALISASI TRAINING HISTORY
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['loss']) + 1)

# Plot 1: Total Loss
axes[0].plot(epochs_range, history['loss'], 'b-o', linewidth=2, markersize=4, label='Total')
axes[0].set_title('Total Loss (Distillation)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# Plot 2: Hard vs Soft Loss
axes[1].plot(epochs_range, history['hard_loss'], 'r-o', linewidth=2, markersize=4, label=f'Hard (α={ALPHA})')
axes[1].plot(epochs_range, history['soft_loss'], 'm-s', linewidth=2, markersize=4, label=f'Soft (α={1-ALPHA:.1f})')
axes[1].set_title('Hard vs Soft Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Accuracy
axes[2].plot(epochs_range, history['accuracy'], 'g-o', linewidth=2, markersize=4)
axes[2].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_ylim([0, 1.05])
axes[2].grid(True, alpha=0.3)

plt.suptitle('📈 Training History — Knowledge Distillation (ResNet50 Student)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Ringkasan
print(f"\n📊 RINGKASAN PERFORMA:")
print(f"{'=' * 45}")
print(f"   Final Loss     : {history['loss'][-1]:.4f}")
print(f"   Final Accuracy : {history['accuracy'][-1]*100:.2f}%")
print(f"   Best Loss      : {min(history['loss']):.4f}")
print(f"   Best Accuracy  : {max(history['accuracy'])*100:.2f}%")
print(f"{'=' * 45}")

---
## 📝 Fase 6 & 7 — Evaluasi & Inferensi (Ditunda)

> ⏳ **STATUS: DITUNDA** — Menunggu Validation/Test Data
>
> **Rencana:**
> - Confusion Matrix, Classification Report, ROC-AUC
> - Perbandingan Student vs Teacher accuracy
> - Inferensi video: frame extraction → batch prediction → threshold

In [ ]:
# ==========================================
# FASE 6: EVALUASI (STUB)
# ==========================================
# Uncomment setelah Validation/Test Data tersedia:

# from sklearn.metrics import (
#     confusion_matrix, classification_report, roc_auc_score, roc_curve
# )
#
# test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
# test_generator = test_datagen.flow_from_directory(
#     'test_data',
#     target_size=(IMG_SIZE, IMG_SIZE),
#     batch_size=BATCH_SIZE,
#     class_mode='binary',
#     shuffle=False,
#     classes=['nsfw', 'safe']
# )
#
# # Student predictions
# y_pred_proba = student_model.predict(test_generator)
# y_pred = (y_pred_proba > 0.5).astype(int).flatten()
# y_true = test_generator.classes
#
# print("CONFUSION MATRIX:")
# print(confusion_matrix(y_true, y_pred))
# print(classification_report(y_true, y_pred, target_names=['nsfw', 'safe']))

print("⏳ Fase 6-7 ditunda — menunggu Validation/Test Data.")

In [ ]:
# ==========================================
# FASE 7: INFERENSI VIDEO (STUB)
# ==========================================

# import cv2
#
# def classify_video(video_path, model, threshold=0.5, sample_rate=30):
#     cap = cv2.VideoCapture(video_path)
#     frames = []
#     frame_idx = 0
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret: break
#         if frame_idx % sample_rate == 0:
#             frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             frame_resized = cv2.resize(frame_rgb, (IMG_SIZE, IMG_SIZE))
#             frames.append(frame_resized)
#         frame_idx += 1
#     cap.release()
#
#     frames_array = preprocess_input(np.array(frames, dtype='float32'))
#     predictions = model.predict(frames_array, batch_size=BATCH_SIZE)
#     nsfw_ratio = np.mean(predictions < 0.5)  # nsfw=low prob in our setup
#     return {
#         'total_frames': len(frames),
#         'nsfw_ratio': float(nsfw_ratio),
#         'verdict': 'nsfw' if nsfw_ratio > 0.3 else 'safe'
#     }

print("⏳ Fase 7 (Inferensi Video) ditunda.")

---
## ✅ Selesai

**Status:**
- ✅ Phase A: Auto-labeling dengan nsfw_detector
- ✅ Fase 2-4: Data handling, preprocessing, arsitektur
- ✅ Phase B: Training dengan Knowledge Distillation
- ⏳ Fase 6-7: Menunggu Val/Test Data

**File output:**
- `best_model.keras` — Model terbaik
- `teacher_predictions.npy` — Cache teacher soft labels

**Langkah selanjutnya:**
1. Pasok Validation Data → aktifkan Early Stopping
2. Pasok Test Data → Confusion Matrix, Student vs Teacher comparison
3. Implementasi inferensi video